In [1]:
import scipy.io
import numpy as np
import os
import glob
import torch
from scipy import signal 
from sklearn.model_selection import LeaveOneOut
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

In [2]:
class SEBlock(nn.Module):
    def __init__(self, C, r=16):
        super(SEBlock, self).__init__()
        self.fc1 = nn.Linear(C, C // r)
        self.fc2 = nn.Linear(C // r, C)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size, channels, _, _ = x.size()
        # Squeeze: Global Average Pooling
        se = x.view(batch_size, channels, -1).mean(dim=2)  # (batch_size, channels)
        se = self.fc1(se)  # (batch_size, channels // r)
        se = nn.ELU()(se)  # Activation
        se = self.fc2(se)  # (batch_size, channels)
        se = self.sigmoid(se).view(batch_size, channels, 1, 1)  # Reshape for scaling
        return x * se.expand_as(x)  # Scale the input

In [3]:
class EEG_SENet(nn.Module):
    def __init__(self, nb_classes=2, n_bands=9, n_channels=27, n_samples=2000, reduction_rate=3, fs=500, dropoutRate=0.5):
        super(EEG_SENet, self).__init__()

        # First Block 
        self.depthwise_conv1 = nn.Conv2d(in_channels=n_bands, out_channels=n_bands, padding='same', 
                                          kernel_size=(n_channels, fs//8), groups=n_bands, stride=1, bias=False)
        self.batch_norm1 = nn.BatchNorm2d(n_bands)
        self.relu1 = nn.ELU()
        
        # SENet
        self.se_block1 = SEBlock(C=n_bands, r=reduction_rate)
        self.dropout1 = nn.Dropout(dropoutRate)

        # Second Block 
        self.depthwise_conv2 = nn.Conv2d(in_channels=n_bands, out_channels=2*n_bands, padding='valid', 
                                          kernel_size=(n_channels, fs//8), groups=n_bands, stride=1, bias=False)
        self.batch_norm2 = nn.BatchNorm2d(2*n_bands)  # Fixed variable name
        self.relu2 = nn.ELU()

        # SENet
        self.se_block2 = SEBlock(C=2*n_bands, r=2*reduction_rate)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.dropout2 = nn.Dropout(dropoutRate)

        # Flatten and Dense Layer
        self.flatten = nn.Flatten()
        self.dense1 = nn.Linear(4356, 2)  # Updated this from 34902 to 4356
        # self.norm_constraint1 = nn.utils.weight_norm(self.dense1)        


    def forward(self, x):
        # First Block
        x = self.depthwise_conv1(x)
        x = self.batch_norm1(x)
        x = self.relu1(x)
        x = self.se_block1(x)
        x = self.dropout1(x)

        # Second Block
        x = self.depthwise_conv2(x)
        # print(f'Shape after depthwise_conv2: {x.shape}')  # Print shape here

        x = self.batch_norm2(x)
        x = self.relu2(x)
        x = self.se_block2(x)
        x = self.pool2(x)
        # print(f'Shape after pool2: {x.shape}')  # Print shape here
        # x = self.dropout2(x)

        # Flatten and Dense Layer
        x = self.flatten(x)
        # print(f'Shape of X after flattening: {x.shape}')

        x = self.dense1(x)
        # print(f'Shape of X after Linear Layer: {x.shape}')
        # x = self.norm_constraint1(x)
        x = F.softmax(x, dim=1)
        
        return x
    
# n_channels = 27
# n_time = 2000
# n_bands = 9
# n_classes = 2

# input_tensor = torch.randn(32, 9, 27, 2000)  # Batch size, channels, height, width
# model = EEG_SENet(2, 9, 27, 2000, reduction_rate=16, fs=500, dropoutRate=0.5).to('cpu') 

# criterion = nn.CrossEntropyLoss().to('cpu')  # Move loss function to GPU if needed
# optimizer = optim.Adam(model.parameters(), lr=1e-3)
# outputs = model(input_tensor)

In [4]:
def load_mat_file(filepath):
    """ Load .mat file and return Xtr, Ytr always; Xte, Yte only if they are not NaN """
    mat_data = scipy.io.loadmat(filepath)
    
    Xtr = mat_data['Xtrain']
    Ytr = mat_data['Ytrain']
    Xte = mat_data['Xtest']
    Yte = mat_data['Ytest']
    Yte_fb = mat_data['Ytest_fb']
    
    # Check if Xte and Yte are NaN or entirely NaN arrays
    if np.isnan(Xte).all() or np.isnan(Yte).all() or np.isnan(Yte_fb).all():
        # If all values in Xte or Yte are NaN, return only Xtr and Ytr
        return Xtr, Ytr
    else:
        # Otherwise, return Xtr, Ytr, Xte, Yte
        return Xtr, Ytr, Xte, Yte, Yte_fb

def create_dataset(current_subject, base_path=None):
    """"
    Create training and test dataset for the current subjects.
    
    Parameters:
    current_subject (int): Subject number (1 to 20).
    base_path (str): The directory where .mat files are stored.

    Returns:
    X_train, Y_train, X_test, Y_test
    """
    Xtr_all = []
    Ytr_all = []
    for sub in range(1, current_subject+1):
        rel_path = f'data/S{sub:02d}_mitrials.mat'
        filepath = os.path.join(base_path, rel_path)

        mat_vars = load_mat_file(filepath)
        if len(mat_vars)==2:
            Xtr, Ytr = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)

        else:
            Xtr, Ytr, Xte, Yte, Yte_fb = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)
            if (sub == current_subject):
                continue
            else:
                # Xtr_all.append(Xte)
                # Ytr_all.append(Yte)
                indx = np.where(Yte==Yte_fb)[0]
                Xtr_all.append(Xte[indx, :, :])
                Ytr_all.append(Yte[indx])
                
    X_train = np.concatenate(Xtr_all, axis=0)
    Y_train = np.concatenate(Ytr_all, axis=0)

    X_test = Xte
    Y_test = Yte
        
    return X_train, Y_train, X_test, Y_test

# Baseline Correction 
def baseline_correction(X, baseline_samples=500):
    n_trails, n_samples, n_channels = X.shape
    Xbc = np.zeros_like(X)
    for t in range(n_trails):
        Xbase = X[t, :baseline_samples-1,:]
        Xeeg = X[t, baseline_samples:, :]
        Xbc[t, baseline_samples:, :] = Xeeg - np.mean(Xbase, axis=0) #baseline correction
    
    Xnew = Xbc[:, baseline_samples:, :]
    return Xnew

# Preprocessing: Surface Laplacian, Bandpass Filter
def bandpass_filtering(X, fs=500, fcut=[0.5, 45], filt_order=5):
    n_trials, n_samples, n_channels = X.shape
    X1 = np.zeros_like(X)
    b,a = signal.butter(filt_order, fcut, fs=fs, btype = 'band', output='ba') 
    for t in range(n_trials):
        for c in range(n_channels):
            #Error here.
            raw_signal = X[t, :, c]
            filt_signal = signal.filtfilt(b, a, raw_signal)
            X1[t, :, c] = filt_signal
            # Xfilt[t, :, c] = signal.filtfilt(b, a, X[t, :, c])
    return X1

# Multi-band Filter Function without averaging over trials
def filter_eeg_multi_band(X, fs=500):
    # Define the frequency bands
    bands = [(4, 8), (8, 12), (12, 16), (16, 20), (20, 24), 
             (24, 28), (28, 32), (32, 36), (36, 40)]
    
    n_trials, n_samples, n_channels = X.shape
    n_bands = len(bands)
    
    # Initialize output array: n_trials x samples x channels x n_bands
    Xout = np.zeros((n_trials, n_samples, n_channels, n_bands))
    
    # Apply bandpass filtering for each band
    for i, (low_cut, high_cut) in enumerate(bands):
        # print(f"Filtering band {low_cut}-{high_cut} Hz")
        X_filt = bandpass_filtering(X, fs=fs, fcut=[low_cut, high_cut], filt_order=5)
        
        # Store the filtered data without averaging across trials
        Xout[:, :, :, i] = X_filt
    
    Xout = np.transpose(Xout, (0, 2, 1, 3))  # (trials, electrodes, samples, n_bands)
    return Xout

In [5]:
def create_dataset(current_subject, base_path=None):
    """"
    Create training and test dataset for the current subjects.
    
    Parameters:
    current_subject (int): Subject number (1 to 20).
    base_path (str): The directory where .mat files are stored.

    Returns:
    X_train, Y_train, X_test, Y_test
    """
    Xtr_all = []
    Ytr_all = []
    for sub in range(1, current_subject+1):
        rel_path = f'data/S{sub:02d}_mitrials.mat'
        filepath = os.path.join(base_path, rel_path)

        mat_vars = load_mat_file(filepath)
        if len(mat_vars)==2:
            Xtr, Ytr = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)

        else:
            Xtr, Ytr, Xte, Yte, Yte_fb = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)
            if (sub == current_subject):
                continue
            else:
                Xtr_all.append(Xte)
                Ytr_all.append(Yte)
                # indx = np.where(Yte==Yte_fb)[0]
                # Xtr_all.append(Xte[indx, :, :])
                # Ytr_all.append(Yte[indx])
                
    X_train = np.concatenate(Xtr_all, axis=0)
    Y_train = np.concatenate(Ytr_all, axis=0)

    X_test = Xte
    Y_test = Yte
        
    return X_train, Y_train, X_test, Y_test



In [6]:
# Function to compute accuracy
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets.squeeze()).sum().item()
            total += targets.size(0)
    
    accuracy = correct / total
    return accuracy

# Function to calculate accuracy
def calculate_accuracy(preds, labels):
    _, predicted = torch.max(preds, 1)
    correct = (predicted == labels).sum().item()
    accuracy = correct / labels.size(0)
    return accuracy


In [ ]:
torch.manual_seed(0)
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 32
num_epochs = 100
fs = 500

perf = dict()
for sub in range(8, 21):
    torch.cuda.empty_cache()
    # print(f'Current Subject : S{sub:02d}')
    Xtr, Ytr, Xte, Yte = create_dataset(sub, base_path=parent_dir)
    
    X_train = baseline_correction(Xtr)
    X_train = filter_eeg_multi_band(X_train)

    X_test = baseline_correction(Xte)
    X_test = filter_eeg_multi_band(X_test)


    # Convert the data to PyTorch tensors
    # X_train shape: (n_trials, electrodes, samples, bands) -> (n_trials, 1, electrodes, samples, bands)
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2).to(device)
    Y_train_tensor = torch.tensor(Ytr, dtype=torch.long).to(device)  # Use long for classification

    # X_test shape: (n_trials, electrodes, samples, bands) -> (n_trials, 1, electrodes, samples, bands)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2).to(device)
    Y_test_tensor = torch.tensor(Yte, dtype=torch.long).to(device)

    # Create TensorDataset and DataLoader for train and test datasets
    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


    n_channels = 27
    n_time = 2000
    n_bands = 9
    n_classes = 2
    model = EEG_SENet(n_classes, n_bands, n_channels, n_time, reduction_rate=8, fs=fs, dropoutRate=0.3).to(device)

    criterion = nn.CrossEntropyLoss().to(device)  # Move loss function to GPU if needed
    optimizer = optim.Adam(model.parameters(), lr=2e-3)

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_accuracy = 0.0
        for i, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets.squeeze())

            accuracy = calculate_accuracy(outputs, targets.squeeze())

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_accuracy += accuracy

            # Print statistics every 100 batches (or any other interval)
            if (i + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], "
                      f"Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")

    
    # Compute accuracy on the test dataset
    accuracy = compute_accuracy(model, test_loader, device)
    perf[f'Sub{sub}'] = accuracy
    
    print(f'Test Subject: S{sub}, Test Accuracy: {accuracy:.4f}')

print(perf)

In [ ]:
mean_acc = list(perf.values())
print(mean_acc)
print(f'Average Accuracy: {np.mean(mean_acc)}')